# Essence of Linear Algebra — Code Companion
*A hands-on notebook to play along with 3Blue1Brown's series.*

**How to use this:**
- Watch a chapter on YouTube, then come here and *touch* the same idea with code.
- Every section has a "🎮 Play with it" cell — change the numbers and re-run.
- The visualizations deliberately mimic Grant's style: vectors as arrows, the grid that morphs under transformations, eigenvectors as the lines that don't tilt.

Playlist: https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab


## 🛠 Setup — run this once

Helper functions for drawing vectors and transformed grids. You won't need to touch this; it's the toolkit the rest of the notebook uses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 3B1B-ish palette on a dark canvas
plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor':   '#0a0a0a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#ddd',
    'grid.color':       '#222',
    'figure.figsize':   (7, 7),
})

BLUE   = '#58c4dd'   # i-hat color
YELLOW = '#fff263'   # j-hat color
RED    = '#fc6255'
GREEN  = '#83c167'
PURPLE = '#9a72ac'
WHITE  = '#f0f0f0'

def setup_axes(ax, lim=5, title=None):
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.25)
    ax.axhline(0, color='#666', lw=0.6)
    ax.axvline(0, color='#666', lw=0.6)
    if title: ax.set_title(title, color=WHITE, fontsize=13)
    return ax

def draw_vector(ax, v, origin=(0,0), color=BLUE, label=None, lw=2.5, alpha=1.0):
    v = np.asarray(v).flatten()
    ax.annotate('', xy=(origin[0]+v[0], origin[1]+v[1]), xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                mutation_scale=18, alpha=alpha))
    if label:
        ox, oy = origin[0]+v[0], origin[1]+v[1]
        # nudge label off the arrowhead
        nudge = 0.18
        ax.text(ox + np.sign(v[0] or 1)*nudge, oy + np.sign(v[1] or 1)*nudge,
                label, color=color, fontsize=13, weight='bold')

def draw_grid(ax, M=None, color=BLUE, alpha=0.5, n=8, lw=0.9):
    """Draw a grid transformed by 2x2 matrix M. M=None draws the standard grid."""
    if M is None: M = np.eye(2)
    M = np.asarray(M, dtype=float)
    for i in range(-n, n+1):
        # vertical line x=i, y from -n to n  -> column points then transform
        pts = np.array([[i]*201, np.linspace(-n, n, 201)])
        tp = M @ pts
        ax.plot(tp[0], tp[1], color=color, alpha=alpha, lw=lw)
        # horizontal line y=i
        pts = np.array([np.linspace(-n, n, 201), [i]*201])
        tp = M @ pts
        ax.plot(tp[0], tp[1], color=color, alpha=alpha, lw=lw)

def show_transformation(M, lim=5, extra_vectors=None, title=None):
    """The iconic 3B1B move: faint original grid, bright transformed grid, basis vectors."""
    fig, ax = plt.subplots()
    setup_axes(ax, lim=lim, title=title)
    draw_grid(ax, M=None, color='#3a6b78', alpha=0.35, lw=0.7)   # original (faint)
    draw_grid(ax, M=M,    color=BLUE,     alpha=0.9,  lw=1.1)    # transformed (bright)
    # basis vectors after transformation
    i_hat = M @ np.array([1, 0])
    j_hat = M @ np.array([0, 1])
    draw_vector(ax, i_hat, color=BLUE,   label='î')
    draw_vector(ax, j_hat, color=YELLOW, label='ĵ')
    if extra_vectors:
        for v, c, lbl in extra_vectors:
            draw_vector(ax, M @ np.asarray(v), color=c, label=lbl)
    plt.show()


---
## 📺 Chapter 1 — Vectors, what even are they?
> *"The fundamental building block."*

Three views of a vector that all describe the same arrow:
- **Physics:** an arrow with length & direction.
- **CS:** an ordered list of numbers.
- **Math:** anything you can add and scale.

Below we just plot a vector. The arrow starts at the origin and ends at the coordinates.

In [ ]:
fig, ax = plt.subplots()
setup_axes(ax, lim=5, title='A vector is an arrow from the origin')

v = np.array([3, 2])
draw_vector(ax, v, color=BLUE, label='v = [3, 2]')
plt.show()


**Vector addition** = "walk along the first arrow, then the second." Tip-to-tail.

In [ ]:
fig, ax = plt.subplots()
setup_axes(ax, lim=6, title='Tip-to-tail addition')

v = np.array([2, 1])
w = np.array([1, 3])

draw_vector(ax, v, color=BLUE,   label='v')
draw_vector(ax, w, origin=v, color=YELLOW, label='w (from v)')
draw_vector(ax, v + w, color=GREEN, label='v + w')
plt.show()


**Scaling** stretches or squishes (or flips, if negative).

In [ ]:
fig, ax = plt.subplots()
setup_axes(ax, lim=6, title='Scaling v by different amounts')

v = np.array([1.5, 1])
for s, c in [(2, BLUE), (1, YELLOW), (-1.5, RED)]:
    draw_vector(ax, s*v, color=c, label=f'{s}·v')
plt.show()


**🎮 Play with it.** Change `v` and `w` below and re-run. Try a negative scalar.

In [ ]:
v = np.array([2, 1])
w = np.array([-1, 2])
s = 1.5

fig, ax = plt.subplots()
setup_axes(ax, lim=6, title=f'{s}·v + w')
draw_vector(ax, s*v, color=BLUE,  label=f'{s}·v')
draw_vector(ax, w, origin=s*v, color=YELLOW, label='w')
draw_vector(ax, s*v + w, color=GREEN, label='result')
plt.show()


---
## 📺 Chapter 2 — Linear combinations, span, basis

A **linear combination** of v and w is anything of the form `a·v + b·w`.
The **span** is the set of *all* such combinations.

If v and w point in different directions, their span is the whole 2D plane.
If they're parallel (linearly dependent), the span collapses to a line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# --- Left: independent vectors, span fills the plane ---
ax = axes[0]; setup_axes(ax, lim=5, title='Independent → span is all of 2D')
v = np.array([2, 1]); w = np.array([-1, 1.5])
# sample many linear combinations as faint dots
A, B = np.meshgrid(np.linspace(-2, 2, 25), np.linspace(-2, 2, 25))
pts = A[..., None]*v + B[..., None]*w
ax.scatter(pts[..., 0], pts[..., 1], s=4, color=BLUE, alpha=0.35)
draw_vector(ax, v, color=BLUE,   label='v')
draw_vector(ax, w, color=YELLOW, label='w')

# --- Right: dependent vectors, span collapses to a line ---
ax = axes[1]; setup_axes(ax, lim=5, title='Parallel → span collapses to a line')
v = np.array([2, 1]); w = np.array([-2, -1])   # w = -v, dependent
A, B = np.meshgrid(np.linspace(-2, 2, 25), np.linspace(-2, 2, 25))
pts = A[..., None]*v + B[..., None]*w
ax.scatter(pts[..., 0], pts[..., 1], s=4, color=RED, alpha=0.35)
draw_vector(ax, v, color=BLUE,   label='v')
draw_vector(ax, w, color=YELLOW, label='w (= -v)')

plt.show()


A **basis** is a minimal set of vectors whose span is the whole space.
The standard basis is `î = [1, 0]` and `ĵ = [0, 1]` — every vector `[x, y]` is just `x·î + y·ĵ`.

---
## 📺 Chapter 3 — Linear transformations and matrices
> *"A matrix is a linear transformation in disguise."*

A **linear transformation** moves every point in space while keeping:
1. The origin fixed.
2. Straight lines straight.
3. Parallel lines parallel.

The columns of the matrix tell you **where î and ĵ land**. Everything else follows.

Below, the faint grid is the original space; the bright blue grid is where the transformation sent it. Watch what î and ĵ do — that's the whole story.

In [ ]:
# Try different transformations:
# Rotation 90°:      [[0, -1], [1, 0]]
# Horizontal shear:  [[1, 1], [0, 1]]
# Scale x2 only:     [[2, 0], [0, 1]]
# Mirror across y:   [[-1, 0], [0, 1]]

M = np.array([[1, 1],
              [0, 1]])    # horizontal shear

show_transformation(M, lim=5, title='Horizontal shear: ĵ slides right')


In [ ]:
# 90° counter-clockwise rotation
M = np.array([[0, -1],
              [1,  0]])
show_transformation(M, lim=5, title='90° rotation')


**🎮 Play with it.** Try a wild matrix. Negative entries flip orientation; large entries stretch hard.

In [ ]:
M = np.array([[1.5, -0.5],
              [0.5,  1.0]])

show_transformation(M, lim=6, title=f'Your transformation\nî → {M[:,0]}, ĵ → {M[:,1]}')


**The big insight:** to find where any vector `v = [x, y]` lands, you don't need to know about "matrix multiplication rules." You just need:

`M·v = x·(new î) + y·(new ĵ)`

That's literally what the formula `[[a,b],[c,d]] @ [x,y] = [ax+by, cx+dy]` is doing.

---
## 📺 Chapter 4 — Matrix multiplication as composition

Multiplying two matrices = applying one transformation, then the other.
Order matters. `A @ B` means "do B first, then A" (right-to-left, like function composition).

In [ ]:
A = np.array([[1, 1],   # shear
              [0, 1]])
B = np.array([[0, -1],  # rotate 90°
              [1,  0]])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Step 1: just B (rotate)
ax = axes[0]; setup_axes(ax, lim=5, title='Step 1: apply B (rotate)')
draw_grid(ax, M=None, color='#3a6b78', alpha=0.35, lw=0.7)
draw_grid(ax, M=B,    color=BLUE,     alpha=0.9)
draw_vector(ax, B @ [1,0], color=BLUE,   label='î')
draw_vector(ax, B @ [0,1], color=YELLOW, label='ĵ')

# Step 2: then A (shear) on top
ax = axes[1]; setup_axes(ax, lim=5, title='Step 2: then apply A (shear)')
draw_grid(ax, M=None,  color='#3a6b78', alpha=0.35, lw=0.7)
draw_grid(ax, M=A @ B, color=BLUE,     alpha=0.9)
draw_vector(ax, (A @ B) @ [1,0], color=BLUE,   label='î')
draw_vector(ax, (A @ B) @ [0,1], color=YELLOW, label='ĵ')

# Step 3: the product A@B does it in one shot
ax = axes[2]; setup_axes(ax, lim=5, title=f'Single matrix A@B does the same\n{A@B}')
draw_grid(ax, M=None,  color='#3a6b78', alpha=0.35, lw=0.7)
draw_grid(ax, M=A @ B, color=GREEN,    alpha=0.9)

plt.show()

print('A @ B (rotate then shear):'); print(A @ B); print()
print('B @ A (shear then rotate) — different result:'); print(B @ A)


---
## 📺 Chapters 5 & 6 — The determinant

The determinant tells you **how much the transformation scales areas** (and whether it flips orientation).

- `det = 2` → areas double.
- `det = 1` → areas preserved (rotations, shears).
- `det = 0` → space gets squished onto a line (or a point). No inverse.
- `det < 0` → orientation flipped (like a mirror).

The unit square (area 1) becomes a parallelogram with area = |det|.

In [ ]:
def show_determinant(M, lim=5):
    M = np.asarray(M, dtype=float)
    fig, ax = plt.subplots()
    setup_axes(ax, lim=lim, title=f'det(M) = {np.linalg.det(M):.2f}')

    # original unit square (faint)
    sq = np.array([[0,1,1,0,0],[0,0,1,1,0]])
    ax.fill(sq[0], sq[1], color=YELLOW, alpha=0.2)
    ax.plot(sq[0], sq[1], color=YELLOW, alpha=0.5, lw=1)

    # transformed unit square (bright)
    tsq = M @ sq
    ax.fill(tsq[0], tsq[1], color=BLUE, alpha=0.4)
    ax.plot(tsq[0], tsq[1], color=BLUE, lw=2)

    draw_vector(ax, M @ [1,0], color=BLUE,   label='î')
    draw_vector(ax, M @ [0,1], color=YELLOW, label='ĵ')
    plt.show()

show_determinant([[2, 0], [0, 1.5]])   # det = 3, areas triple


In [ ]:
show_determinant([[1, 2], [3, 1]])   # det = -5, flipped + scaled


In [ ]:
show_determinant([[2, 1], [4, 2]])   # det = 0, squished onto a line!


When det = 0, you've lost a dimension. Two columns are linearly dependent. You **cannot un-do** this transformation — there's no inverse.

---
## 📺 Chapter 7 — Inverse, column space, rank, null space

- **Column space** = span of the columns = the set of all possible outputs.
- **Rank** = dimension of the column space.
- **Null space** (kernel) = all input vectors that get sent to zero.

When `det ≠ 0`, the transformation is invertible and you can solve `Ax = b` uniquely.

In [ ]:
A = np.array([[2, 1],
              [1, 3]])
b = np.array([4, 5])

x = np.linalg.solve(A, b)
print(f'A = \n{A}')
print(f'b = {b}')
print(f'x (solution to Ax = b) = {x}')
print(f'Check: A @ x = {A @ x}')
print(f'rank(A) = {np.linalg.matrix_rank(A)}, det(A) = {np.linalg.det(A):.2f}')


In [ ]:
# A rank-deficient case — null space is non-trivial
A = np.array([[1, 2],
              [2, 4]])    # rows are dependent, det = 0
print(f'rank(A) = {np.linalg.matrix_rank(A)}')
print(f'det(A)  = {np.linalg.det(A):.2f}')

# Vectors in the null space get mapped to 0
v_null = np.array([2, -1])
print(f'A @ [2, -1] = {A @ v_null}  ← null space vector')

# Visualize: every input lands on a single line (the column space)
fig, ax = plt.subplots()
setup_axes(ax, lim=6, title='Rank-1 transform: outputs collapse to a line')
inputs = np.random.uniform(-4, 4, (2, 200))
outputs = A @ inputs
ax.scatter(inputs[0], inputs[1], s=8, color='#3a6b78', alpha=0.6, label='inputs')
ax.scatter(outputs[0], outputs[1], s=10, color=BLUE, alpha=0.9, label='outputs (column space)')
ax.legend(facecolor='#111', edgecolor='#444')
plt.show()


---
## 📺 Chapter 9 — Dot products and duality

`v · w = |v||w|cos(θ)` — geometrically it's the **length of v's projection onto w**, scaled by |w|.

Sign tells you the angle:
- positive → same general direction (angle < 90°)
- zero → perpendicular
- negative → opposite direction (angle > 90°)

In [ ]:
def show_projection(v, w):
    v = np.asarray(v, dtype=float); w = np.asarray(w, dtype=float)
    # projection of v onto w
    proj = (v @ w) / (w @ w) * w

    fig, ax = plt.subplots()
    setup_axes(ax, lim=5, title=f'v·w = {v @ w:.2f}    (projection length = {(v@w)/np.linalg.norm(w):.2f})')

    # the line through w (the "screen" we project onto)
    t = np.linspace(-5, 5, 2)
    ax.plot(t*w[0], t*w[1], color='#444', lw=1, ls='--')

    draw_vector(ax, v, color=BLUE,   label='v')
    draw_vector(ax, w, color=YELLOW, label='w')
    draw_vector(ax, proj, color=GREEN, label='proj_w(v)')
    # dashed line from v down to projection
    ax.plot([v[0], proj[0]], [v[1], proj[1]], color='#888', lw=1, ls=':')
    plt.show()

show_projection([3, 4], [4, 1])


In [ ]:
# 🎮 Try perpendicular vectors — dot product should be 0
show_projection([2, 3], [3, -2])


---
## 📺 Chapter 13 — Change of basis

The vector itself doesn't change. The *coordinates describing it* depend on the basis you choose.

If Jennifer uses basis `b1, b2` instead of `î, ĵ`, then her coordinates `[x_J, y_J]` mean `x_J·b1 + y_J·b2` in our language. The matrix `B = [b1 | b2]` translates Jennifer → us. `B⁻¹` translates us → Jennifer.

In [ ]:
# Jennifer's basis
b1 = np.array([2, 1])
b2 = np.array([-1, 1])
B  = np.column_stack([b1, b2])

# A vector that Jennifer calls [1, 2] in HER coordinates
v_jen = np.array([1, 2])
v_ours = B @ v_jen
print(f"Jennifer's coords: {v_jen}  →  our coords: {v_ours}")

# Going back
v_jen_again = np.linalg.inv(B) @ v_ours
print(f'Round trip: {v_jen_again}')

fig, ax = plt.subplots()
setup_axes(ax, lim=5, title="Same vector, two coordinate systems")
# Jennifer's grid
draw_grid(ax, M=B, color=PURPLE, alpha=0.4, lw=0.9, n=4)
# our grid (faint)
draw_grid(ax, M=None, color='#3a6b78', alpha=0.3, lw=0.7)
draw_vector(ax, b1, color=PURPLE, label='b₁')
draw_vector(ax, b2, color=PURPLE, label='b₂')
draw_vector(ax, v_ours, color=GREEN,
            label=f"v: Jen says [1,2], we say {v_ours.tolist()}")
plt.show()


---
## 📺 Chapter 14 — Eigenvectors and eigenvalues
> *"Most vectors get knocked off their span. Eigenvectors don't."*

An **eigenvector** of `A` is a vector `v` such that `A @ v = λ·v` — the transformation only stretches it (by factor λ, the **eigenvalue**), never rotates it off its line.

Geometrically: find the lines through the origin that the transformation leaves invariant.

In [ ]:
A = np.array([[3, 1],
              [0, 2]])

vals, vecs = np.linalg.eig(A)
print(f'Eigenvalues: {vals}')
print(f'Eigenvectors (columns):\n{vecs}')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Before
ax = axes[0]; setup_axes(ax, lim=5, title='Before: eigenvector lines (dashed)')
draw_grid(ax, M=None, color='#3a6b78', alpha=0.35)
for i, color in enumerate([RED, GREEN]):
    e = vecs[:, i]
    t = np.linspace(-5, 5, 2)
    ax.plot(t*e[0], t*e[1], color=color, lw=1, ls='--', alpha=0.6)
    draw_vector(ax, e, color=color, label=f'v{i+1}')

# After: eigenvectors stay on their lines, just scaled by λ
ax = axes[1]; setup_axes(ax, lim=5, title='After A: eigenvectors stayed on their lines')
draw_grid(ax, M=A, color=BLUE, alpha=0.7)
for i, color in enumerate([RED, GREEN]):
    e = vecs[:, i]
    t = np.linspace(-5, 5, 2)
    ax.plot(t*e[0], t*e[1], color=color, lw=1, ls='--', alpha=0.6)
    draw_vector(ax, A @ e, color=color, label=f'A·v{i+1} = {vals[i]:.1f}·v{i+1}')

plt.show()


**Why this matters everywhere:**
- PCA: eigenvectors of the covariance matrix = directions of maximum variance.
- Stability of systems: eigenvalues tell you whether things blow up or settle down.
- Google PageRank: it's literally an eigenvector problem.

---
## 🌉 Where this shows up in your world

A quick taste — these are all just linear algebra wearing different costumes.

**1. Covariance matrix → PCA.** The eigenvectors of a returns covariance matrix point in the directions where a portfolio varies most. The first eigenvector is roughly "the market factor." Below, we generate two correlated "asset returns" and recover the principal axes.

In [ ]:
rng = np.random.default_rng(42)
# Two correlated synthetic return streams
n = 500
asset1 = rng.normal(0, 1, n)
asset2 = 0.7 * asset1 + rng.normal(0, 0.6, n)
X = np.column_stack([asset1, asset2])

cov = np.cov(X.T)
vals, vecs = np.linalg.eig(cov)
# sort by eigenvalue descending
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]

fig, ax = plt.subplots()
setup_axes(ax, lim=4, title='Returns scatter — principal axes are eigenvectors of cov')
ax.scatter(X[:,0], X[:,1], s=10, color='#3a6b78', alpha=0.5)
# scale eigenvectors by sqrt(eigenvalue) for visibility
for i, color in enumerate([RED, GREEN]):
    v = vecs[:, i] * np.sqrt(vals[i]) * 2.5
    draw_vector(ax, v, color=color, label=f'PC{i+1} (λ={vals[i]:.2f})')
plt.show()

print(f'Variance explained: PC1 = {vals[0]/vals.sum():.1%}, PC2 = {vals[1]/vals.sum():.1%}')


**2. Neural networks.** Every layer is `output = activation(W @ input + b)`. `W` is a matrix — a linear transformation. Training a network = finding the right transformations.

**3. Rotations on charts.** When you rotate a price chart or transform indicators, you're applying 2x2 matrices.

---

## 🧭 What to do next

After each video, come back here and:
1. Re-run the matching cell.
2. Change the matrix to something weird.
3. Predict what the picture will look like *before* you hit run. That's where the intuition gets baked in.

When you hit eigenvectors (Ch 14) and want to go deeper, the natural next steps are SVD, change of basis tricks for diagonalization, and the spectral theorem — say the word and I'll add chapters.
